# Basic Network Sniffer

An educational packet-capture demo for Google Colab using Python, Scapy, Pandas, and Gradio.

**Note:** This notebook captures test traffic generated inside the Colab environment itself.

In [ ]:
!pip install -q scapy gradio pandas

In [ ]:
from scapy.all import *
import pandas as pd
import gradio as gr
import socket
import time
import threading

In [ ]:
def capture_packets(packet_count=10):
    captured = []

    def process_packet(packet):
        try:
            if IP in packet:
                source = packet[IP].src
                destination = packet[IP].dst

                if TCP in packet:
                    protocol = "TCP"
                elif UDP in packet:
                    protocol = "UDP"
                elif ICMP in packet:
                    protocol = "ICMP"
                else:
                    protocol = "Other"

                payload = ""
                if Raw in packet:
                    payload = bytes(packet[Raw].load)[:80]
                    payload = payload.decode("utf-8", errors="replace")

                captured.append({
                    "Source IP": source,
                    "Destination IP": destination,
                    "Protocol": protocol,
                    "Packet Size": len(packet),
                    "Payload Preview": payload
                })
        except Exception:
            pass

    sniff(iface="lo", prn=process_packet, count=packet_count, timeout=10, store=False)
    return captured

In [ ]:
def generate_test_traffic():
    def send_packets():
        time.sleep(1)
        for i in range(10):
            try:
                sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
                message = f"Network Sniffer Test Packet {i+1}"
                sock.sendto(message.encode(), ("127.0.0.1", 9999))
                sock.close()
            except Exception:
                pass
            time.sleep(0.2)

    thread = threading.Thread(target=send_packets)
    thread.start()

In [ ]:
def run_sniffer():
    captured = []

    def capture():
        nonlocal captured
        captured = capture_packets(10)

    capture_thread = threading.Thread(target=capture)
    capture_thread.start()
    generate_test_traffic()
    capture_thread.join()

    if not captured:
        return pd.DataFrame({"Message": ["No packets captured. Run the capture again."]})

    return pd.DataFrame(captured)

# Test the sniffer
result = run_sniffer()
result

In [ ]:
def start_capture():
    return run_sniffer()

with gr.Blocks(title="Basic Network Sniffer") as app:
    gr.Markdown("""
# 🛡️ Basic Network Sniffer

Capture and analyze test network packets generated inside the Colab environment.

**Displayed:** Source IP, Destination IP, Protocol, Packet Size, and Payload Preview.
""")

    capture_button = gr.Button("🚀 Capture Packets", variant="primary")
    output_table = gr.Dataframe(
        headers=["Source IP", "Destination IP", "Protocol", "Packet Size", "Payload Preview"],
        label="Captured Packets"
    )

    capture_button.click(fn=start_capture, inputs=None, outputs=output_table)

app.launch(share=True)